In [1]:
"""
my prediction_file is like /niddk-data-central/leo_workspace/iwatch_W/val_result/zeroshot_prediction/CHAP_ALL_ADULTS/i0001A.csv
each csv file looks like:
segment,timestamp,prediction
0,2013-04-23 12:12:00,sitting
0,2013-04-23 12:12:10,sitting
0,2013-04-23 12:12:20,not-sitting

my ground truth file is in root = "/niddk-data-central/iWatch/pre_processed_seg/W/10s_val.h5"
help to align the two files with f['subject_id'] and f['timestamp']

"""

'\nmy prediction_file is like /niddk-data-central/leo_workspace/iwatch_W/val_result/zeroshot_prediction/CHAP_ALL_ADULTS/i0001A.csv\neach csv file looks like:\nsegment,timestamp,prediction\n0,2013-04-23 12:12:00,sitting\n0,2013-04-23 12:12:10,sitting\n0,2013-04-23 12:12:20,not-sitting\n\nmy ground truth file is in root = "/niddk-data-central/iWatch/pre_processed_seg/W/10s_val.h5"\nhelp to align the two files with f[\'subject_id\'] and f[\'timestamp\']\n\n'

In [1]:
import os
import h5py
import pandas as pd
from utils import compute_accuracy_from_confusion_matrix, compute_additional_metrics_from_confusion_matrix
from sklearn.metrics import confusion_matrix
import numpy as np

def compute_metrics(predictions_dir: str, h5_path: str):
    # load ground truth
    with h5py.File(h5_path, 'r') as f:
        subj_arr  = f['subject_id'][:].astype(str)
        ts_arr    = f['timestamp'][:]
        gt_labels = f['y'][:]
    
    gt_labels = gt_labels.reshape(-1)
    ts_arr = ts_arr.reshape(-1)
    subj_arr = np.repeat(subj_arr, 42)

    if ts_arr.dtype.kind in ('S', 'U'):
        gt_ts = pd.to_datetime(ts_arr.astype(str))
    else:
        gt_ts = pd.to_datetime(ts_arr, unit='s')

    gt_df = pd.DataFrame({
        'subject_id': subj_arr,
        'timestamp':  gt_ts,
        'ground':     gt_labels
    }).set_index(['subject_id', 'timestamp'])

    mapping = {'sitting': 0, 'not-sitting': 1}

    all_true = []
    all_pred = []
    total_pred = 0
    total_matched = 0

    per_subject_results = {}

    for fname in os.listdir(predictions_dir):
        if not fname.endswith('.csv'):
            continue

        subject_id = os.path.splitext(fname)[0]
        pred_df = pd.read_csv(
            os.path.join(predictions_dir, fname),
            parse_dates=['timestamp']
        )
        pred_df['subject_id'] = subject_id
        pred_df['pred'] = pred_df['prediction'].map(mapping)
        pred_df = pred_df.dropna(subset=['pred'])

        orig_n = len(pred_df)
        total_pred += orig_n

        merged = (
            pred_df
            .set_index(['subject_id', 'timestamp'])
            .join(gt_df, how='inner')
            .reset_index()
        )
        after_n = len(merged)
        total_matched += after_n

        if orig_n != after_n:
            print(f"Warning: subject {subject_id} dropped {orig_n - after_n} rows out of {orig_n}")

        # append to all
        all_pred.extend(merged['pred'].astype(int).tolist())
        all_true.extend(merged['ground'].tolist())

        # per-subject metrics
        subj_true = merged['ground'].tolist()
        subj_pred = merged['pred'].astype(int).tolist()

        if len(subj_true) == 0:
            continue

        subj_cm = confusion_matrix(subj_true, subj_pred, labels=[0, 1])
        acc, bal_acc = compute_accuracy_from_confusion_matrix(subj_cm)
        metrics = compute_additional_metrics_from_confusion_matrix(subj_cm)

        per_subject_results[subject_id] = {
            'accuracy': acc,
            'balanced_accuracy': bal_acc,
            **metrics
        }

    if total_pred != total_matched:
        print(f"Overall warning: dropped {total_pred - total_matched} rows out of {total_pred}")
    else:
        print("All prediction rows matched ground truth.")

    # overall metrics
    cm = confusion_matrix(all_true, all_pred)
    val_acc, val_balanced_acc = compute_accuracy_from_confusion_matrix(cm)
    additional_metrics = compute_additional_metrics_from_confusion_matrix(cm)

    print(f"Overall Accuracy: {val_acc:.4f}, Balanced Accuracy: {val_balanced_acc:.4f}")
    for k, v in additional_metrics.items():
        print(f"{k}: {v}")

    # print("\nPer-Subject Metrics:")
    # for sid, metrics in per_subject_results.items():
    #     summary = f"Subject {sid}: acc={metrics['accuracy']:.4f}, bal_acc={metrics['balanced_accuracy']:.4f}"
    #     for k, v in metrics.items():
    #         if k in ['accuracy', 'balanced_accuracy']:
    #             continue
    #         if isinstance(v, (float, int)):
    #             summary += f", {k}={v:.4f}"
    #         elif isinstance(v, list) or isinstance(v, np.ndarray):
    #             formatted = ", ".join([f"{x:.4f}" for x in v])
    #             summary += f", {k}=[{formatted}]"
    #         else:
    #             summary += f", {k}={v}"
    #     print(summary)

    return val_acc, val_balanced_acc, additional_metrics, per_subject_results


## HIP

In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/H/CHAP-FT/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/H/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/H/CHAP-ZS/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/H/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/H/shallow-moca/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/H/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


# Wrist

In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/W/CHAP-FT/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/W/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/W/CHAP-FT/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/W/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


In [ ]:
PRED_DIR = '/niddk-data-central/leo_workspace/complete_test_prediction/W/shallow-moca/predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/W/10s_test_complete.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


# Validation

In [2]:
import pandas as pd

PRED_DIR = '/niddk-data-central/iWatch/submit_result/W/CHAP-FT/val_predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/W/10s_val.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


df = pd.DataFrame.from_dict(per_subject_results, orient='index').reset_index()
df = df.rename(columns={'index': 'subject_id'})
df[df['subject_id']=='i0195A'] 

Class 0:
  TP = 405, FP = 42, FN = 99, TN = 0
  ⚠️ Specificity for class 0 is 0 — all negative samples predicted as class 0

Full Confusion Matrix:
 [[405  99]
 [ 42   0]]
All prediction rows matched ground truth.


/DeepPostures_MAE/MSSE_2021_pt/utils.py:132: RuntimeWarning: invalid value encountered in divide
  f1_score = 2 * ppv * sensitivity / (ppv + sensitivity)


Overall Accuracy: 0.8783, Balanced Accuracy: 0.8601
sensitivity: [0.8989062216212944, 0.82120889188027]
specificity: [0.82120889188027, 0.8989062216212944]
positive_predictive_value: [0.9332162423573696, 0.7450744243246682]
negative_predictive_value: [0.7450744243246682, 0.9332162423573696]
f1_score: [0.9157399713896575, 0.7812912721952695]


,subject_id,accuracy,balanced_accuracy,sensitivity,specificity,positive_predictive_value,negative_predictive_value,f1_score
19,i0195A,0.741758,0.401786,"[0.8035714285714286, 0.0]","[0.0, 0.8035714285714286]","[0.9060402684563759, 0.0]","[0.0, 0.9060402684563759]","[0.8517350157728706, 0.0]"


In [17]:
import pandas as pd

PRED_DIR = '/niddk-data-central/iWatch/submit_result/W/CHAP-ZS/val_predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/W/10s_val.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


df = pd.DataFrame.from_dict(per_subject_results, orient='index').reset_index()
df = df.rename(columns={'index': 'subject_id'})
df[df['subject_id']=='i0195A'] 

All prediction rows matched ground truth.
Overall Accuracy: 0.6784, Balanced Accuracy: 0.7319
sensitivity: [0.6182370391298206, 0.8455045151158226]
specificity: [0.8455045151158226, 0.6182370391298206]
positive_predictive_value: [0.9175052410901468, 0.44347199341021415]
negative_predictive_value: [0.44347199341021415, 0.9175052410901468]
f1_score: [0.7387121276057051, 0.5817911657436174]


,subject_id,accuracy,balanced_accuracy,sensitivity,specificity,positive_predictive_value,negative_predictive_value,f1_score
19,i0195A,0.708791,0.798611,"[0.6924603174603174, 0.9047619047619048]","[0.9047619047619048, 0.6924603174603174]","[0.9886685552407932, 0.19689119170984457]","[0.19689119170984457, 0.9886685552407932]","[0.8144690781796966, 0.32340425531914896]"


In [16]:
import pandas as pd

PRED_DIR = '/niddk-data-central/iWatch/submit_result/H/CHAP-FT/val_predictions'
GT_H5    = '/niddk-data-central/iWatch/pre_processed_long_seg/H/10s_val.h5'
val_acc, val_balanced_acc, additional_metrics, per_subject_results = compute_metrics(PRED_DIR, GT_H5)


df = pd.DataFrame.from_dict(per_subject_results, orient='index').reset_index()
df = df.rename(columns={'index': 'subject_id'})
df[df['subject_id']=='i0195A'] 

All prediction rows matched ground truth.
Overall Accuracy: 0.9288, Balanced Accuracy: 0.9229
sensitivity: [0.9349811618449151, 0.9108259393795034]
specificity: [0.9108259393795034, 0.9349811618449151]
positive_predictive_value: [0.9683568822460451, 0.8275740797518351]
negative_predictive_value: [0.8275740797518351, 0.9683568822460451]
f1_score: [0.9513763943864699, 0.8672065465953547]


,subject_id,accuracy,balanced_accuracy,sensitivity,specificity,positive_predictive_value,negative_predictive_value,f1_score
19,i0195A,0.915751,0.954365,"[0.9087301587301587, 1.0]","[1.0, 0.9087301587301587]","[1.0, 0.4772727272727273]","[0.4772727272727273, 1.0]","[0.9521829521829522, 0.6461538461538462]"
